# 02U — Model Development (Updated)
**Fixes:** Label Leakage | Cross-Domain Negative Pairs | Vocabulary OOV
**Skip:** CV synthetic length improvement

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from src.preprocessing.nlp_preprocessor import NLPPreprocessor
from src.preprocessing.embeddings import EmbeddingManager
from src.models.model_architecture import SkillAlignMatcher
from src.models.custom_loss import focal_loss
from src.training.train import ModelTrainer
from src.utils.metrics import compute_all_metrics, compute_classification_report, check_performance_targets
from src.utils.visualization import TrainingVisualizer

print(f'TF: {tf.__version__}')
print('Imports OK!')

TF: 2.21.0
Imports OK!


## 1. Load Raw Data

In [2]:
# Load job postings
df_posting = pd.read_csv(
    '../Dataset/database_design/job_posting.csv',
    usecols=['job_posting_id', 'title', 'job_description']
).dropna(subset=['job_description'])
print(f'Job postings: {len(df_posting):,}')

# Load job_skills
df_job_skills = pd.read_csv('../Dataset/database_design/job_skills.csv')
df_skill_names = pd.read_csv('../Dataset/database_design/skills.csv')
df_job_skills = df_job_skills.merge(df_skill_names, on='skill_id', how='left')
job_skills_grouped = (
    df_job_skills.groupby('job_posting_id')['skill_name']
    .apply(list).reset_index()
)
job_skills_grouped.columns = ['job_posting_id', 'required_skills']

# Load industries untuk cross-domain grouping
df_job_ind = pd.read_csv('../Dataset/database_design/job_industries.csv')
df_industries = pd.read_csv('../Dataset/database_design/industries.csv')
df_job_ind = df_job_ind.merge(df_industries, on='industry_id', how='left')
ind_per_job = df_job_ind.groupby('job_posting_id')['industry_name'].first().reset_index()

# Gabungkan semua
df = df_posting.merge(job_skills_grouped, on='job_posting_id', how='inner')
df = df.merge(ind_per_job, on='job_posting_id', how='left')
df['industry_name'] = df['industry_name'].fillna('Other')
df = df.dropna(subset=['job_description'])
print(f'Combined shape: {df.shape}')
print(f'Unique industries: {df["industry_name"].nunique()}')

Job postings: 123,842
Combined shape: (122090, 5)
Unique industries: 375


## 2. FIX #1 — Extract Keywords dari Job Description (bukan 35 generic skills)

In [3]:
SAMPLE_SIZE = min(len(df), 30000)
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Fit TF-IDF pada job descriptions asli
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5
)
tfidf.fit(df_sample['job_description'].str[:2000])
feature_names = np.array(tfidf.get_feature_names_out())
print(f'TF-IDF vocab: {len(feature_names)}')

def extract_top_keywords(text, tfidf_model, feat_names, topn=10):
    try:
        vec = tfidf_model.transform([str(text)[:2000]])
        scores = vec.toarray()[0]
        top_idx = scores.argsort()[::-1][:topn]
        return [feat_names[i] for i in top_idx if scores[i] > 0]
    except:
        return []

# Test
sample_kw = extract_top_keywords(df_sample['job_description'].iloc[0], tfidf, feature_names, 6)
print(f'Sample keywords: {sample_kw}')

TF-IDF vocab: 5000
Sample keywords: ['leasing', 'resident', 'marketing', 'property', 'occupancy', 'property management']


## 3. FIX #2 — Cross-Domain Negative Pairing

**Sebelumnya:** cv_neg ↔ job_desc (SAMA) → label leakage

**Fix:** cv_pos dari job A (domain X) dipasangkan dengan job B dari **domain berbeda Y**

In [4]:
# =============================================================================
# CELL 7 BARU — GANTI SELURUH ISI Cell 7 di notebook 02U_model_development.ipynb
# DENGAN KODE DI BAWAH INI.
#
# Cell ini menggantikan logic sintesis CV-Job pairs yang lama (yang punya
# masalah lexical leakage + binary labels).
# =============================================================================

from src.preprocessing.pair_synthesizer import (
    CVJobPairSynthesizer, PairSynthConfig
)

# Synthesizer otomatis hitung industry index + role index dari df_sample.
# Catatan: Cell 5 sebelumnya yang menyiapkan df_sample, tfidf, feature_names
#          tetap dipertahankan — TIDAK perlu diubah.
synth = CVJobPairSynthesizer(
    df_sample=df_sample,
    tfidf=tfidf,
    feature_names=feature_names,
    config=PairSynthConfig(
        seed=42,
        # Bobot soft label (boleh disesuaikan)
        w_skill_overlap=0.50,
        w_seniority=0.25,
        w_domain=0.15,
        w_role=0.10,
        # Distribusi mode (default sudah seimbang)
        p_high_match=0.30,
        p_partial=0.25,
        p_seniority_mismatch=0.15,
        p_same_domain_diff_role=0.15,
        p_cross_domain=0.15,
    )
)

cv_texts, job_texts, labels = synth.synthesize()

# Sanity check distribusi label
print(f'Pairs: {len(labels):,}')
print(f'Label  mean: {labels.mean():.3f} | std: {labels.std():.3f}')
print(f'Label range: [{labels.min():.3f}, {labels.max():.3f}]')

import numpy as np
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
counts, _ = np.histogram(labels, bins=bins)
print('\nDistribusi label:')
for lo, hi, ct in zip(bins[:-1], bins[1:], counts):
    pct = ct / len(labels) * 100
    bar = '#' * int(pct / 2)
    print(f'  [{lo:.1f}, {hi:.1f}): {ct:6d} ({pct:5.1f}%) {bar}')

Pairs: 29,970
Label  mean: 0.578 | std: 0.258
Label range: [0.004, 1.000]

Distribusi label:
  [0.0, 0.2):   4525 ( 15.1%) #######
  [0.2, 0.4):   3121 ( 10.4%) #####
  [0.4, 0.6):   4660 ( 15.5%) #######
  [0.6, 0.8):   9480 ( 31.6%) ###############
  [0.8, 1.0):   8184 ( 27.3%) #############


## 4. FIX #3 — Vocabulary dari Job Descriptions Asli (bukan hanya CV synthetic)

In [5]:
MAX_VOCAB_SIZE = 15000  # Naik dari 10000
MAX_SEQ_LEN   = 300
EMBEDDING_DIM = 128

preprocessor = NLPPreprocessor(
    max_vocab_size=MAX_VOCAB_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    use_lemmatizer=True,
    language='english'
)

# FIX #3: Fit pada CV + Job descriptions asli untuk pastikan kata teknikal masuk vocab
extra_job_texts = df_sample['job_description'].str[:2000].tolist()
all_fit_texts   = cv_texts + job_texts + extra_job_texts
preprocessor.fit(all_fit_texts)
print(f'Vocabulary size: {preprocessor.vocab_size}')

# Cek kata teknikal
word_index = preprocessor.word_index
tech_check = ['python', 'react', 'docker', 'tensorflow', 'javascript', 'kubernetes']
print('Tech terms in vocab:')
for w in tech_check:
    idx = word_index.get(w)
    print(f'  {w}: {"idx=" + str(idx) if idx else "NOT FOUND"}')

Vocabulary size: 15000
Tech terms in vocab:
  python: idx=1163
  react: idx=2148
  docker: idx=3286
  tensorflow: idx=7987
  javascript: idx=1797
  kubernetes: idx=2703


In [6]:
cv_sequences  = preprocessor.transform(cv_texts)
job_sequences = preprocessor.transform(job_texts)
print(f'CV: {cv_sequences.shape} | Job: {job_sequences.shape}')

CV: (29970, 300) | Job: (29970, 300)


## 5. Word2Vec Embedding

In [7]:
tokenized = [t.split() for t in preprocessor.preprocess_batch(all_fit_texts)]
emb_manager = EmbeddingManager(embedding_dim=EMBEDDING_DIM, min_count=2, window=5)
emb_manager.train_word2vec(tokenized, epochs=20, sg=1)

embedding_matrix = emb_manager.create_embedding_matrix(
    word_index=preprocessor.word_index,
    vocab_size=preprocessor.vocab_size
)
print(f'Embedding matrix: {embedding_matrix.shape}')

for w in ['python', 'react', 'docker']:
    sim = emb_manager.get_similar_words(w, topn=3)
    if sim:
        print(f'  "{w}": {[(x[0], round(x[1],2)) for x in sim]}')

Embedding matrix: (15000, 128)
  "python": [('java', 0.81), ('javascript', 0.79), ('sql', 0.77)]
  "react": [('j', 0.75), ('angular', 0.75), ('typescript', 0.75)]
  "docker": [('kubernetes', 0.83), ('aws', 0.71), ('runtimes', 0.71)]


## 6. Train-Val-Test Split

In [8]:
# =============================================================================
# CELL 14 BARU — GANTI SELURUH ISI Cell 14 di notebook
#
# Fix: stratify=labels error untuk continuous labels.
# Solusi: bin labels jadi 10 bucket, stratify pakai bucket-nya.
# =============================================================================

import numpy as np
from sklearn.model_selection import train_test_split

# Bin label kontinyu ke 10 bucket untuk stratification
# (sklearn stratify butuh discrete classes, bukan continuous floats)
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)[1:-1]  # 9 cutoffs untuk 10 bins
label_bins = np.digitize(labels, bin_edges)

print(f'Label bins distribution:')
unique, counts = np.unique(label_bins, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  bin {u}: {c} samples')

# ----- Split 1: train+val vs test (80/20) -----
(cv_train, cv_test,
 job_train, job_test,
 y_train, y_test,
 bins_train, _) = train_test_split(
    cv_sequences, job_sequences, labels, label_bins,
    test_size=0.2, random_state=42, stratify=label_bins
)

# ----- Split 2: train vs val (85/15 dari sisa) -----
cv_train, cv_val, job_train, job_val, y_train, y_val = train_test_split(
    cv_train, job_train, y_train,
    test_size=0.15, random_state=42, stratify=bins_train
)

print(f'\nTrain: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}')
print(f'y_train mean: {y_train.mean():.3f}, std: {y_train.std():.3f}')
print(f'y_val   mean: {y_val.mean():.3f}, std: {y_val.std():.3f}')
print(f'y_test  mean: {y_test.mean():.3f}, std: {y_test.std():.3f}')

Label bins distribution:
  bin 0: 835 samples
  bin 1: 3690 samples
  bin 2: 883 samples
  bin 3: 2238 samples
  bin 4: 2075 samples
  bin 5: 2585 samples
  bin 6: 6878 samples
  bin 7: 2602 samples
  bin 8: 7684 samples
  bin 9: 500 samples

Train: 20,379 | Val: 3,597 | Test: 5,994
y_train mean: 0.577, std: 0.258
y_val   mean: 0.578, std: 0.258
y_test  mean: 0.578, std: 0.258


## 7. Build & Train Model

In [9]:
matcher = SkillAlignMatcher(
    vocab_size=preprocessor.vocab_size,
    max_seq_len=MAX_SEQ_LEN,
    embedding_dim=EMBEDDING_DIM,
    attention_units=128,
    embedding_matrix=embedding_matrix,
    trainable_embedding=True
)
model = matcher.build_model()
model.summary()

Model: "SkillAlign_Matcher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cv_input            │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_input           │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 300, 128)  │  1,920,000 │ cv_input[0][0],   │
│ (Embedding)         │                   │            │ job_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_conv1d_1         │ (None, 300, 128)  │     49,280 │ shared_embedding… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_conv1d_1        │ (None, 300, 128)  │     49,280 │ shared_embedding… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_bn_1             │ (None, 300, 128)  │        512 │ cv_conv1d_1[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_bn_1            │ (None, 300, 128)  │        512 │ job_conv1d_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_conv1d_2         │ (None, 300, 64)   │     24,640 │ cv_bn_1[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_conv1d_2        │ (None, 300, 64)   │     24,640 │ job_bn_1[0][0]    │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_bn_2             │ (None, 300, 64)   │        256 │ cv_conv1d_2[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_bn_2            │ (None, 300, 64)   │        256 │ job_conv1d_2[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ custom_attention    │ (None, 128)       │     24,960 │ cv_bn_2[0][0],    │
│ (CustomAttentionLa… │                   │            │ job_bn_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_global_pool      │ (None, 64)        │          0 │ cv_bn_2[0][0]     │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_global_pool     │ (None, 64)        │          0 │ job_bn_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_merge       │ (None, 256)       │          0 │ custom_attention… │
│ (Concatenate)       │                   │            │ cv_global_pool[0… │
│                     │                   │            │ job_global_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ feature_merge[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_bn_1          │ (None, 256)       │      1,024 │ dense_1[0][0]   

 Total params: 2,202,881 (8.40 MB)

 Trainable params: 2,201,345 (8.40 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [10]:
# =============================================================================
# CELL 17 BARU — GANTI SELURUH ISI Cell 17 di notebook
#
# Bypass ModelTrainer karena F1ScoreCallback-nya tidak kompatibel dengan
# label kontinyu (regression). Kita compile + fit langsung.
# =============================================================================

import os
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.metrics import MeanAbsoluteError, RootMeanSquaredError
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, TensorBoard, ModelCheckpoint
)

# ----- Compile model untuk REGRESSION (label sekarang kontinyu [0,1]) -----
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=Huber(delta=0.1),
    metrics=[
        MeanAbsoluteError(name='mae'),
        RootMeanSquaredError(name='rmse'),
    ]
)
print('Compiled (regression mode).')

# ----- Setup direktori -----
log_dir = '../logs/training_v3'
model_dir = '../models'
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# ----- Callbacks (monitoring val_mae untuk regression) -----
callbacks = [
    EarlyStopping(
        monitor='val_mae', mode='min',
        patience=8, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_mae', mode='min',
        factor=0.5, patience=4, min_lr=1e-7, verbose=1
    ),
    TensorBoard(
        log_dir=log_dir,
        histogram_freq=1, write_graph=True, profile_batch=0
    ),
    ModelCheckpoint(
        filepath=os.path.join(model_dir, 'best_model_v3.keras'),
        monitor='val_mae', mode='min',
        save_best_only=True, verbose=1
    ),
]

# ----- Train -----
print('Training...')
history = model.fit(
    x=[cv_train, job_train],
    y=y_train,
    validation_data=([cv_val, job_val], y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
print('Training complete!')

Compiled (regression mode).
Training...
Epoch 1/50
318/319 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step - loss: 0.0211 - mae: 0.2571 - rmse: 0.3211
Epoch 1: val_mae improved from None to 0.21038, saving model to ../models\best_model_v3.keras

Epoch 1: finished saving model to ../models\best_model_v3.keras
319/319 ━━━━━━━━━━━━━━━━━━━━ 43s 125ms/step - loss: 0.0197 - mae: 0.2422 - rmse: 0.3030 - val_loss: 0.0166 - val_mae: 0.2104 - val_rmse: 0.2620 - learning_rate: 0.0010
Epoch 2/50
318/319 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - loss: 0.0173 - mae: 0.2178 - rmse: 0.2736
Epoch 2: val_mae improved from 0.21038 to 0.18737, saving model to ../models\best_model_v3.keras

Epoch 2: finished saving model to ../models\best_model_v3.keras
319/319 ━━━━━━━━━━━━━━━━━━━━ 39s 121ms/step - loss: 0.0165 - mae: 0.2100 - rmse: 0.2656 - val_loss: 0.0143 - val_mae: 0.1874 - val_rmse: 0.2373 - learning_rate: 0.0010
Epoch 3/50
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 0.0140 - mae: 0.1842 - rmse: 0.2359
Epoch 3: 

## 8. Evaluation

In [11]:
# =============================================================================
# CELL 19 BARU — GANTI SELURUH ISI Cell 19 di notebook
#
# Evaluasi untuk regression (label kontinyu).
# Tidak lagi pakai accuracy/precision/recall sebagai metric utama,
# tapi MAE & RMSE. Pseudo-accuracy@0.5 dipertahankan sebagai sanity check.
# =============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('../logs/plots_v3', exist_ok=True)

# ----- Predict on test set -----
y_pred = model.predict([cv_test, job_test], verbose=0).flatten()

# ----- Regression metrics -----
mae = float(np.mean(np.abs(y_pred - y_test)))
mse = float(np.mean((y_pred - y_test) ** 2))
rmse = float(np.sqrt(mse))
correlation = float(np.corrcoef(y_pred, y_test)[0, 1])

# ----- Pseudo-classification untuk perbandingan dgn versi lama -----
y_pred_bin = (y_pred > 0.5).astype(int)
y_true_bin = (y_test > 0.5).astype(int)
acc = float(np.mean(y_pred_bin == y_true_bin))

print('=== Regression Metrics ===')
print(f'  MAE : {mae:.4f}  (target ≤ 0.02)')
print(f'  RMSE: {rmse:.4f}')
print(f'  MSE : {mse:.6f}')
print(f'  Korelasi y_pred vs y_test: {correlation:.3f}')
print()
print('=== Pseudo-Classification (threshold 0.5) ===')
print(f'  Accuracy: {acc:.4f}  (target ≥ 0.85)')
print()
print('=== Distribusi prediksi vs ground truth ===')
print(f'  y_pred  mean={y_pred.mean():.3f}, std={y_pred.std():.3f}')
print(f'  y_test  mean={y_test.mean():.3f}, std={y_test.std():.3f}')

# ----- Plot training history -----
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
h = history.history

axes[0].plot(h['loss'], label='train')
axes[0].plot(h['val_loss'], label='val')
axes[0].set_title('Huber Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(h['mae'], label='train MAE')
axes[1].plot(h['val_mae'], label='val MAE')
axes[1].axhline(0.02, color='red', linestyle='--', label='target 0.02')
axes[1].set_title('MAE')
axes[1].set_xlabel('epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../logs/plots_v3/training_history_v3.png', dpi=100, bbox_inches='tight')
plt.show()

# ----- Plot prediction vs actual scatter -----
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([0, 1], [0, 1], 'r--', label='perfect prediction')
plt.xlabel('y_test (actual)')
plt.ylabel('y_pred')
plt.title(f'Pred vs Actual (corr={correlation:.3f}, MAE={mae:.4f})')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('../logs/plots_v3/pred_vs_actual_v3.png', dpi=100, bbox_inches='tight')
plt.show()

=== Regression Metrics ===
  MAE : 0.1449  (target ≤ 0.02)
  RMSE: 0.2152
  MSE : 0.046313
  Korelasi y_pred vs y_test: 0.593

=== Pseudo-Classification (threshold 0.5) ===
  Accuracy: 0.8115  (target ≥ 0.85)

=== Distribusi prediksi vs ground truth ===
  y_pred  mean=0.595, std=0.208
  y_test  mean=0.578, std=0.258


In [12]:
# =============================================================================
# CELL 20 BARU — GANTI SELURUH ISI Cell 20 di notebook
#
# Tambahan visualisasi untuk regression. Confusion matrix tetap dibuat
# untuk perbandingan visual (di-threshold 0.5), tapi metrik utama tetap
# di Cell 19.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Threshold ke binary HANYA untuk visualisasi (perbandingan dgn versi lama)
y_pred_bin = (y_pred > 0.5).astype(int)
y_true_bin = (y_test > 0.5).astype(int)

# ----- Confusion matrix -----
cm = confusion_matrix(y_true_bin, y_pred_bin)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['<0.5', '≥0.5'])
ax.set_yticklabels(['<0.5', '≥0.5'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix @ threshold 0.5')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('../logs/plots_v3/cm_v3.png', dpi=100, bbox_inches='tight')
plt.show()

# ----- Distribusi skor (overlapping histogram) -----
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_pred[y_true_bin == 1], bins=30, alpha=0.5,
        label='actual high (y_test ≥ 0.5)', color='green')
ax.hist(y_pred[y_true_bin == 0], bins=30, alpha=0.5,
        label='actual low (y_test < 0.5)', color='red')
ax.axvline(0.5, color='black', linestyle='--', label='threshold 0.5')
ax.set_xlabel('Predicted score'); ax.set_ylabel('Count')
ax.set_title('Score distribution')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../logs/plots_v3/score_dist_v3.png', dpi=100, bbox_inches='tight')
plt.show()

print('Plots saved to ../logs/plots_v3/')

Plots saved to ../logs/plots_v3/


## 9. Sanity Check — Verifikasi High vs Low Match

In [13]:
quick_tests = [
    {
        'label': 'HIGH MATCH (Data Scientist)',
        'cv':  'Experienced Data Scientist with 5 years in Python TensorFlow machine learning deep learning and data analysis.',
        'job': 'Looking for a Data Scientist with strong Python skills experience in ML frameworks TensorFlow and statistical analysis.'
    },
    {
        'label': 'LOW MATCH (Marketing vs Data Analyst)',
        'cv':  'Marketing Manager with 7 years experience in digital marketing SEO SEM social media management and campaign analytics.',
        'job': 'Data Analyst position requiring SQL Python Tableau statistical analysis and business intelligence.'
    },
    {
        'label': 'VERY LOW (Designer vs Developer)',
        'cv':  'UI/UX Designer with expertise in Figma Adobe XD user research prototyping and design systems. No coding experience.',
        'job': 'Frontend Developer position requiring React TypeScript JavaScript CSS and component-based architecture.'
    }
]

print('=== Sanity Check (skor harus turun dari high ke very low) ===')
for t in quick_tests:
    cv_seq  = preprocessor.process(t['cv'])
    job_seq = preprocessor.process(t['job'])
    score   = model.predict([cv_seq, job_seq], verbose=0)[0][0]
    print(f"  [{t['label']}] Score: {score:.4f}")

=== Sanity Check (skor harus turun dari high ke very low) ===
  [HIGH MATCH (Data Scientist)] Score: 0.6813
  [LOW MATCH (Marketing vs Data Analyst)] Score: 0.4947
  [VERY LOW (Designer vs Developer)] Score: 0.8773


## 10. Save Artifacts

In [14]:
# =============================================================================
# CELL 24 BARU — GANTI SELURUH ISI Cell 24 di notebook
#
# Karena Cell 17 sudah bypass ModelTrainer, variabel `trainer` tidak ada lagi.
# Save model langsung pakai model.save().
# =============================================================================

import os
import json
import joblib

os.makedirs('../models', exist_ok=True)
os.makedirs('../preprocessors', exist_ok=True)

# ----- Save model -----
model_path = '../models/skillalign_matcher_v3.keras'
model.save(model_path)
print(f'Model: {model_path}')

# ----- Save preprocessor -----
preprocessor_path = '../preprocessors/nlp_preprocessor_v3.pkl'
joblib.dump(preprocessor, preprocessor_path)
print(f'Preprocessor: {preprocessor_path}')

# ----- Save embedding manager -----
emb_path = '../preprocessors/embedding_manager_v3.pkl'
emb_manager.save_model(emb_path)
print(f'Embedding: {emb_path}')

# ----- Save config + metrics -----
config = matcher.get_model_config()
config['version'] = 'v3'
config['fixes'] = [
    'pair_synthesizer_with_5_modes',
    'soft_continuous_labels',
    'huber_regression_loss',
    'cross_domain_negative_pairs',
    'hard_negative_same_domain_diff_role',
    'hard_negative_seniority_mismatch',
]
config['training_summary'] = {
    'total_epochs': len(history.history.get('loss', [])),
    'final_loss': float(history.history['loss'][-1]) if 'loss' in history.history else None,
    'final_mae': float(history.history['mae'][-1]) if 'mae' in history.history else None,
    'best_val_loss': float(min(history.history['val_loss'])) if 'val_loss' in history.history else None,
    'best_val_mae': float(min(history.history['val_mae'])) if 'val_mae' in history.history else None,
}
config['evaluation_metrics'] = {
    'mae': mae,
    'rmse': rmse,
    'mse': mse,
    'pseudo_accuracy_at_0.5': acc,
    'pred_test_correlation': correlation,
}

config_path = '../models/model_config_v3.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f'Config: {config_path}')

print('\n=== Done! ===')

Model: ../models/skillalign_matcher_v3.keras
Preprocessor: ../preprocessors/nlp_preprocessor_v3.pkl
Embedding: ../preprocessors/embedding_manager_v3.pkl
Config: ../models/model_config_v3.json

=== Done! ===
